# 01. Сбор данных


## 0. Зависимости

In [1]:
!pip install telethon pandas tqdm vk-api


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Импорты и ключи

In [1]:
import sys, os, re
sys.path.insert(0, os.path.abspath('..'))
from src.collectors.telegram_collector import TelegramCollector
from src.collectors.vk_collector import VKCollector
import pandas as pd, asyncio
print('OK')

OK


In [2]:
TG_API_ID = 34601398
TG_API_HASH = 'c00cdcf1d521291ca3482c89de087651'
VK_TOKEN = 'vk1.a.WR2Nvxm5NsPerBkkjlFIa1YV9HyX7nuqvu4MtRQiWC5IoRZfZBvv1MzNCkQVHhCGNjfJ_6syhmTLEdc8coNvxRSOfYjgJhkZ1mdNmvJpHCz_9KrL9gZudU5Lt0_rD6uCScj44qmkLViIrXql8vW-qMJzRvUZY9yG4gNVMxYlZQ4xuaqFl7BF4gfk0CyybnaHT0C8inkc5JK1iATM-kO0Ig'

## 2. Списки каналов

In [3]:
with open("../data/raw/channel_list.txt") as f:
    TG_CHANNELS = [l.strip() for l in f if l.strip().startswith("@")]
print(f"TG: {len(TG_CHANNELS)}")

vk_path = "../data/raw/vk_channel_list.txt"
if os.path.exists(vk_path):
    with open(vk_path) as f:
        VK_GROUPS = [l.strip() for l in f if l.strip()]
    print(f"VK: {len(VK_GROUPS)} (from file)")
else:
    print("VK not found, searching via API...")
    from src.collectors.channel_finder import VKFinder
    finder = VKFinder(access_token=VK_TOKEN)
    queries = {
        "Blogs": ["blog","lifestyle","blogger"],
        "Beauty": ["beauty","makeup","cosmetics"],
        "Travel": ["travel","trip"],
        "Food": ["food","cooking","recipes"],
        "Sport": ["fitness","sport","workout"],
        "Fashion": ["fashion","style"],
        "Tech": ["tech","gadgets"],
        "Business": ["business","startup"],
        "Marketing": ["smm","marketing","digital"],
        "Auto": ["auto","cars"],
        "Humor": ["fun","memes"],
        "Psychology": ["psychology","motivation"],
    }
    df_vk = finder.collect_all_queries(queries=queries, per_query=15, delay=0.35)
    df_vk = df_vk[df_vk["username"] != ""]
    df_vk = df_vk.drop_duplicates(subset=["username"])
    VK_GROUPS = sorted(df_vk["username"].unique().tolist())
    with open(vk_path, "w") as f:
        f.write(" ".join(VK_GROUPS))
    print(f"VK: {len(VK_GROUPS)} (found and saved)")


TG: 270
VK: 379 (from file)


## 3. Telegram (batched + resume)

In [4]:
TG_OUT = '../data/raw/telegram_posts_raw.csv'

async def tg_collect():
    c = TelegramCollector(api_id=TG_API_ID, api_hash=TG_API_HASH)
    await c.start()
    try:
        df = await c.collect(TG_CHANNELS, TG_OUT, limit=200, delay=2.5, batch=25, rest=300)
        return df
    finally:
        await c.stop()

df_tg = await tg_collect()
print(f'TG: {len(df_tg)} posts')
df_tg.head(2)

[TG] @rokvelll
[RESUME] 60040 постов из 499 каналов
Осталось: 110/270


TG:   0%|                                                                                                                        | 0/110 [00:00<?, ?it/s]

[SKIP] @marinlya: user profile, not a channel


TG:   0%|                                                                                                                        | 0/110 [00:08<?, ?it/s]


CancelledError: 

## 4. VK (batched + resume)

In [5]:
VK_OUT = "../data/raw/vk_posts_raw.csv"
vc = VKCollector(token=VK_TOKEN)
df_vk = vc.collect(VK_GROUPS, VK_OUT, limit=200, delay=1.2, batch=10, rest=100)
print(f"VK: {len(df_vk)} posts")
df_vk.head(2)


[RESUME] 7445 постов из 52 групп
Осталось: 368/379


VK:   0%|                                                                                                                        | 0/368 [00:00<?, ?it/s]

[OK] 20englishclub21: 200


VK:   0%|▎                                                                                                             | 1/368 [00:13<1:20:26, 13.15s/it]

[OK] a_trip: 200


...

[OK] b_0_s: 200


VK:   5%|█████                                                                                                        | 17/368 [04:59<1:20:00, 13.68s/it]

[VK] bashkortauto: [9] Flood control
[FLOOD_VK] ждём 180s (попытка 1/3)
[VK] bashkortauto: [9] Flood control
[FLOOD_VK] ждём 360s (попытка 2/3)


VK:   5%|█████                                                                                                        | 17/368 [14:00<4:49:10, 49.43s/it]


KeyboardInterrupt: 

## 5. Объединение

In [8]:
# Читаем из сохранённых CSV (работает даже если сбор не завершён)
df_tg = pd.read_csv("../data/raw/telegram_posts_raw.csv")
df_vk = pd.read_csv("../data/raw/vk_posts_raw.csv")

COLS = ["platform","channel_username","channel_title","followers_count",
        "post_id","post_text","text_length","has_media","media_type",
        "has_photo","has_video","has_document","has_link","has_hashtag",
        "n_links_raw","n_hashtags_raw","n_mentions_raw",
        "views_count","likes_count","forwards_count","comments_count",
        "reactions_count","replies_count",
        "is_advert","country","city","published_at","collected_at"]

df_all = pd.concat([df_tg[COLS], df_vk[COLS]], ignore_index=True)
df_all = df_all.drop_duplicates(subset=["platform","channel_username","post_id"])

# Классификация рекламных маркеров
def detect_ad_type(text):
    if not isinstance(text, str) or not text: return "none"
    t = text.lower()
    if re.search(r"erid:", t): return "erid"
    if re.search(r"#реклама", t): return "hashtag_reklama"
    if re.search(r"#спонсор|спонсорский пост", t): return "sponsor"
    if re.search(r"на правах рекламы", t): return "text_marker"
    if re.search(r"#промо|#сотрудничество|#интеграция", t): return "hashtag_other"
    return "other"

df_all["ad_type"] = df_all["post_text"].apply(detect_ad_type)
df_all.to_csv("../data/raw/all_posts_raw.csv", index=False)

print(f"Всего: {len(df_all)} постов")
print(f"TG: {(df_all.platform=="Telegram").sum()}, VK: {(df_all.platform=="VK").sum()}")
print(f"Рекламных: {df_all.is_advert.sum()} ({df_all.is_advert.mean():.2%})")
print(f"Типы рекламы:")
print(df_all[df_all["ad_type"]!="none"]["ad_type"].value_counts())


Всего: 70476 постов
TG: 60040, VK: 10436
Рекламных: 1145 (1.62%)
Типы рекламы:
ad_type
other              67375
erid                 169
hashtag_reklama       76
hashtag_other          5
text_marker            2
Name: count, dtype: int64
